##### ARTI 560 - Computer Vision

## Visual Representations with DINOv2 - Exercise

### Exercise 1: Unsupervised Clustering

In this exercise, you will use the `KMeans` algorithm from sklearn to group 20 images from the Oxford Pet dataset into 2 clusters (Cats vs. Dogs) based purely on their CLS tokens.

Instructions:

1.  Extract the 384-dimensional [CLS] tokens from 20 images of the Oxford-IIIT Pet dataset. Ensure your selection includes a mix of both cats and dogs.

2. Apply K-Means Clustering ($n=2$) to group the vectors based on mathematical similarity rather than provided labels.

3. Compare the predicted clusters against ground-truth labels.

In [3]:
import torch
import numpy as np
from torchvision import datasets, transforms, models
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# Image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load dataset and ensure 10 cats and 10 dogs
dataset = datasets.OxfordIIITPet(root='./data', download=True, transform=transform)

cat_indices = []
dog_indices = []

for i in range(len(dataset)):
    _, label = dataset[i]
    if label <= 11 and len(cat_indices) < 10:  # Oxford Pet cat labels are 0-11
        cat_indices.append(i)
    elif label > 11 and len(dog_indices) < 10: # Oxford Pet dog labels are 12-36
        dog_indices.append(i)
    if len(cat_indices) == 10 and len(dog_indices) == 10:
        break

selected_indices = cat_indices + dog_indices
images = torch.stack([dataset[i][0] for i in selected_indices])
# Create ground truth: 0 for cats, 1 for dogs
binary_true = np.array([0]*10 + [1]*10)

#  Feature extraction using ViT [CLS] tokens
model = models.vit_b_16(weights='IMAGENET1K_V1')
model.eval()

with torch.no_grad():
    # Process inputs through the ViT backbone
    x = model._process_input(images)
    n = x.shape[0]
    batch_class_token = model.class_token.expand(n, -1, -1)
    x = torch.cat([batch_class_token, x], dim=1)
    x = model.encoder(x)
    
    # Extract the [CLS] token (index 0)
    cls_tokens = x[:, 0].numpy()

# Apply K-Means Clustering (n=2)
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
predicted_clusters = kmeans.fit_predict(cls_tokens)

#  Output results
print("--- Clustering Results ---")
print(f"Predicted Labels: {predicted_clusters}")
print(f"Ground Truth:     {binary_true}")

# Calculate ARI (1.0 is perfect clustering)
ari = adjusted_rand_score(binary_true, predicted_clusters)
print(f"\nAdjusted Rand Index (ARI): {ari:.2f}")

--- Clustering Results ---
Predicted Labels: [0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1]
Ground Truth:     [0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1]

Adjusted Rand Index (ARI): 1.00


K-Means succeeded in separating cats from dogs 100%

### Exercise 2: Image Classification with DINOv2

In this exercise you'll use a DINOv2 model with a pre-trained linear head to classify an image. You will observe how the model maps visual features to specific ImageNet-1k categories.

Instructions:
1. For this exercise, you must use the following Model ID. This specific checkpoint includes the necessary classification head trained on ImageNet-1k:

    Model ID: `facebook/dinov2-small-imagenet1k-1-layer`

2. Find an image online to make the inference. To ensure the model has a fair chance of success, the image should belong to one of the ImageNet-1k classes (e.g., a Golden Retriever, a grand piano, a school bus, or a coffee mug).

In [6]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import requests

# Load the model and processor
model_id = "facebook/dinov2-small-imagenet1k-1-layer"
processor = AutoImageProcessor.from_pretrained(model_id)
model = AutoModelForImageClassification.from_pretrained(model_id)

#  Prepare the image
# A Golden Retriever
url = "https://images.unsplash.com/photo-1552053831-71594a27632d"
image = Image.open(requests.get(url, stream=True).raw)

# Preprocess and Inference
inputs = processor(images=image, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

predicted_class_idx = logits.argmax(-1).item()
predicted_label = model.config.id2label[predicted_class_idx]

print(f"Predicted Class: {predicted_label}")

Loading weights:   0%|          | 0/225 [00:00<?, ?it/s]

Predicted Class: golden retriever
